In [ ]:
# =========================
# Reproducible XAI Script (SHAP SUMMARY ONLY) - LOCAL ipynb (ONE CELL)
# - Runs ONLY F / OF winners (NO O, NO OFR)
# - SHAP summary dot plot ONLY (no bar, no waterfall, no LIME)
# - Output folder: ./results (shallow)
# - Runs THREE variants:
#     (1) orig13  : ORIGINAL 13 features only
#     (2) edgeON  : winner feature list as-is (includes edges)
#     (3) edgeOFF : winner feature list but removing edge_GES_* (with fallback policy)
# - ✅ NO TITLE on the plot
# - ✅ If feature starts with edge_GES__ (or edge_GES_), ONLY the prefix is removed in SHAP display names
#   (training/features remain unchanged)
# =========================

import os
import json
import random
import warnings
from typing import List, Tuple, Dict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import shap
import xgboost as xgb
import lightgbm as lgb

from sklearn.metrics import f1_score

warnings.filterwarnings("ignore")

# -------------------------
# Config
# -------------------------
RANDOM_STATE = 42
BASE_DIR = os.getcwd()
print("[CWD]", BASE_DIR)

RESULTS_F_PATH     = os.path.join(BASE_DIR, "results_F.csv")
RESULTS_OF_PATH    = os.path.join(BASE_DIR, "results_OF.csv")
FEATURES_USED_PATH = os.path.join(BASE_DIR, "features_used.csv")

DATA_DAG = {
    "NOTEARS": {
        "train": os.path.join(BASE_DIR, "data_with_features_NOTEARS_train.csv"),
        "val":   os.path.join(BASE_DIR, "data_with_features_NOTEARS_val.csv"),
        "test":  os.path.join(BASE_DIR, "data_with_features_NOTEARS_test.csv"),
    },
    "PC": {
        "train": os.path.join(BASE_DIR, "data_with_features_PC_train.csv"),
        "val":   os.path.join(BASE_DIR, "data_with_features_PC_val.csv"),
        "test":  os.path.join(BASE_DIR, "data_with_features_PC_test.csv"),
    },
    "GES": {
        "train": os.path.join(BASE_DIR, "data_with_features_GES_train.csv"),
        "val":   os.path.join(BASE_DIR, "data_with_features_GES_val.csv"),
        "test":  os.path.join(BASE_DIR, "data_with_features_GES_test.csv"),
    },
    "GOLEM": {
        "train": os.path.join(BASE_DIR, "data_with_features_GOLEM_train.csv"),
        "val":   os.path.join(BASE_DIR, "data_with_features_GOLEM_val.csv"),
        "test":  os.path.join(BASE_DIR, "data_with_features_GOLEM_test.csv"),
    },
}

OUT_DIR = os.path.join(BASE_DIR, "results")
os.makedirs(OUT_DIR, exist_ok=True)

# edgeOFF 후 feature가 비는 경우 처리 정책: "orig13" / "skip" / "error"
EDGE_OFF_EMPTY_POLICY = "orig13"

# 원본 13개 변수명 (orig13 variant)
ORIG_FEATURES_13 = [
    "leverage_ratio",
    "asset_liabilities",
    "roe",
    "asset_turnover",
    "debt_ratio",
    "debt_ratio2",
    "roa",
    "capitalization_ratio",
    "longtermdebt_invcap",
    "totaldebt_invcap",
    "cash_debt",
    "debt_ebitda",
    "rect_turn",
]

# -------------------------
# Global plot font (Times New Roman)
# -------------------------
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["axes.unicode_minus"] = False

# -------------------------
# Determinism
# -------------------------
UINT32_MAX = 2**32 - 1

def seed_uint32(x: int) -> int:
    return int(x) % UINT32_MAX

def set_seed_everywhere(seed: int):
    s = seed_uint32(seed)
    random.seed(s)
    np.random.seed(s)

set_seed_everywhere(RANDOM_STATE)

# -------------------------
# Index-like column removal
# -------------------------
def is_index_col_name(col: str) -> bool:
    c = str(col).strip()
    cl = c.lower()
    if c.startswith("Unnamed") or cl.startswith("unnamed"):
        return True
    if cl in {"index", "_index"}:
        return True
    if cl.endswith("_index"):
        return True
    return False

def looks_like_index_series(s: pd.Series) -> bool:
    try:
        v = pd.to_numeric(s, errors="coerce")
        if v.isna().mean() > 0.3:
            return False
        n = len(v)
        if n <= 5:
            return False
        if v.nunique(dropna=True) / float(n) < 0.98:
            return False
        vv = v.to_numpy()
        if np.all(vv == np.arange(n)):
            return True
        if np.all(vv == np.arange(1, n + 1)):
            return True
        if np.all(np.diff(vv) >= 0):
            dif = np.diff(vv)
            if np.mean(np.abs(dif - 1.0) < 1e-9) > 0.95:
                return True
    except Exception:
        return False
    return False

def drop_all_indexlike_cols(df: pd.DataFrame, name: str) -> pd.DataFrame:
    if df is None or df.shape[1] == 0:
        return df

    drop_cols = []
    for c in list(df.columns):
        if is_index_col_name(c):
            drop_cols.append(c)

    for c in list(df.columns):
        if c in drop_cols:
            continue
        try:
            if looks_like_index_series(df[c]):
                drop_cols.append(c)
        except Exception:
            pass

    if drop_cols:
        drop_cols = list(dict.fromkeys(drop_cols))
        print(f"[DROP] {name}: {drop_cols}")
        df = df.drop(columns=drop_cols)

    bad = [c for c in df.columns if str(c).lower().startswith("unnamed")]
    if bad:
        raise RuntimeError(f"[FATAL] {name}: Unnamed columns still exist after drop: {bad}")
    return df

def read_csv_safely(path: str, name: str) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    return drop_all_indexlike_cols(df, name=name)

def sanitize_feature_list(cols: List[str]) -> List[str]:
    cols2 = [c for c in cols if (not is_index_col_name(c)) and (not str(c).lower().startswith("unnamed"))]
    out, seen = [], set()
    for c in cols2:
        if c not in seen:
            out.append(c)
            seen.add(c)
    return out

# -------------------------
# Target & data prep
# -------------------------
TARGET_CANDIDATES = ["label", "target", "y", "failure", "bank_failure", "default", "is_failed"]

def detect_target_col(df: pd.DataFrame) -> str:
    for c in TARGET_CANDIDATES:
        if c in df.columns:
            return c
    raise ValueError(f"Target column not found among {TARGET_CANDIDATES}")

def make_xy(df_tr, df_va, df_te, target_col: str, feat_cols: List[str]):
    feat_cols = sanitize_feature_list(feat_cols)
    missing = [c for c in feat_cols if (c not in df_tr.columns) or (c not in df_va.columns) or (c not in df_te.columns)]
    if missing:
        raise ValueError(f"Missing features in dataset. Example: {missing[:20]} (total {len(missing)})")

    for c in feat_cols:
        df_tr[c] = pd.to_numeric(df_tr[c], errors="coerce")
        df_va[c] = pd.to_numeric(df_va[c], errors="coerce")
        df_te[c] = pd.to_numeric(df_te[c], errors="coerce")

    df_tr[feat_cols] = df_tr[feat_cols].replace([np.inf, -np.inf], np.nan)
    df_va[feat_cols] = df_va[feat_cols].replace([np.inf, -np.inf], np.nan)
    df_te[feat_cols] = df_te[feat_cols].replace([np.inf, -np.inf], np.nan)

    med = df_tr[feat_cols].median(axis=0, skipna=True)
    df_tr[feat_cols] = df_tr[feat_cols].fillna(med)
    df_va[feat_cols] = df_va[feat_cols].fillna(med)
    df_te[feat_cols] = df_te[feat_cols].fillna(med)

    X_tr = df_tr[feat_cols].to_numpy(np.float32)
    X_va = df_va[feat_cols].to_numpy(np.float32)
    X_te = df_te[feat_cols].to_numpy(np.float32)
    y_tr = df_tr[target_col].to_numpy(np.int64)
    y_va = df_va[target_col].to_numpy(np.int64)
    y_te = df_te[target_col].to_numpy(np.int64)
    return X_tr, y_tr, X_va, y_va, X_te, y_te

# -------------------------
# Model (deterministic, CPU, 1 thread)
# -------------------------
def normalize_model_name(x: str) -> str:
    s = str(x).strip()
    m = {"XGB": "XGBoost", "XGBOOST": "XGBoost", "LGB": "LightGBM", "LGBM": "LightGBM", "LIGHTGBM": "LightGBM"}
    return m.get(s.upper(), s)

LGBM_DET_PARAMS = dict(
    n_estimators=2000,
    learning_rate=0.02,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbose=-1,
    deterministic=True,
    force_col_wise=True,
)

XGB_DET_PARAMS = dict(
    tree_method="hist",
    device="cpu",
    n_estimators=800,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbosity=0,
)

def best_f1_threshold(y_true: np.ndarray, y_prob: np.ndarray, n_grid: int = 101) -> float:
    thresholds = np.linspace(0.0, 1.0, n_grid)
    best_t, best_f1 = 0.5, -1.0
    for t in thresholds:
        pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return float(best_t)

def train_model_and_threshold(model_name: str, X_tr, y_tr, X_va, y_va):
    model_name = normalize_model_name(model_name)

    if model_name == "LightGBM":
        clf = lgb.LGBMClassifier(**LGBM_DET_PARAMS)
        clf.fit(X_tr, y_tr)
        thr = best_f1_threshold(y_va, clf.predict_proba(X_va)[:, 1], n_grid=101)
        return clf, thr

    if model_name == "XGBoost":
        clf = xgb.XGBClassifier(**XGB_DET_PARAMS)
        clf.fit(X_tr, y_tr)
        thr = best_f1_threshold(y_va, clf.predict_proba(X_va)[:, 1], n_grid=101)
        return clf, thr

    raise ValueError(f"Only LightGBM/XGBoost supported. got={model_name}")

# -------------------------
# Winner selection + features_used loader
# -------------------------
def pick_best_row(df: pd.DataFrame) -> pd.Series:
    need = ["AUPRC", "ECE", "Brier", "N_FEAT", "FEATURE_KEY", "DAG", "MODEL"]
    for c in need:
        if c not in df.columns:
            raise ValueError(f"Missing column in results: {c}")
    return df.sort_values(["AUPRC", "ECE", "Brier", "N_FEAT"], ascending=[False, True, True, True]).iloc[0]

def load_feature_list(features_used_df: pd.DataFrame, feature_key: str) -> List[str]:
    fk = str(feature_key).strip()
    row = features_used_df.loc[features_used_df["FEATURE_KEY"] == fk]
    if row.empty:
        raise ValueError(f"FEATURE_KEY not found: {fk}")
    cols = json.loads(row["FEATURES_JSON"].iloc[0])
    cols = sanitize_feature_list(cols)
    if len(cols) == 0:
        raise ValueError(f"All features removed after sanitize. FEATURE_KEY={fk}")
    return cols

def filter_edge_features_with_policy(
    feat_cols: List[str],
    include_edges: bool,
    df_tr_cols: List[str],
    policy: str,
) -> Tuple[List[str], str]:
    """
    returns: (final_feat_cols, policy_tag)
    policy_tag: "as_is" or "fallback_orig13" or "skipped"
    """
    feat_cols = sanitize_feature_list(feat_cols)

    if include_edges:
        return feat_cols, "as_is"

    out = [c for c in feat_cols if not (str(c).startswith("edge_GES__") or str(c).startswith("edge_GES_"))]
    out = sanitize_feature_list(out)

    if len(out) > 0:
        return out, "as_is"

    if policy == "skip":
        return [], "skipped"
    if policy == "error":
        raise ValueError("[FATAL] After excluding edge_GES_*, feature list became empty (policy=error).")

    dfcols = set(df_tr_cols)
    avail = [c for c in ORIG_FEATURES_13 if c in dfcols]
    if len(avail) == 0:
        raise ValueError("[FATAL] Fallback orig13 failed: none of ORIG_FEATURES_13 exist in train columns.")
    return avail, "fallback_orig13"

# -------------------------
# SHAP display-name prettifier (ONLY for plotting labels)
# -------------------------
def prettify_feature_names_for_shap(feat_cols: List[str]) -> List[str]:
    pretty = []
    for c in feat_cols:
        if c.startswith("edge_GES__"):
            pretty.append(c.replace("edge_GES__", "", 1))
        elif c.startswith("edge_GES_"):
            pretty.append(c.replace("edge_GES_", "", 1))
        else:
            pretty.append(c)
    return pretty

# -------------------------
# SHAP summary dot only (NO TITLE)
# -------------------------
def shap_positive_class(shap_values):
    if isinstance(shap_values, (list, tuple)):
        if len(shap_values) == 2:
            return shap_values[1]
        return shap_values[-1]
    arr = np.array(shap_values)
    if arr.ndim == 3:
        return arr[:, :, -1]
    if arr.ndim == 2:
        return arr
    raise ValueError(f"Unsupported shap_values shape: {arr.shape}")

def run_shap_summary_only(
    model,
    X_background,
    X_explain,
    feature_names,
    out_path: str,
    max_background: int = 2000,
    max_explain: int = 5000,
):
    rng = np.random.RandomState(RANDOM_STATE)

    Xb = X_background
    if Xb is not None and Xb.shape[0] > max_background:
        idx = rng.choice(Xb.shape[0], size=max_background, replace=False)
        Xb = Xb[idx]

    Xe = X_explain
    if Xe.shape[0] > max_explain:
        idx = rng.choice(Xe.shape[0], size=max_explain, replace=False)
        Xe = Xe[idx]

    explainer = shap.TreeExplainer(model, data=Xb, feature_perturbation="interventional")
    sv = explainer.shap_values(Xe, check_additivity=False)
    sv_pos = shap_positive_class(sv)

    fig = plt.figure(figsize=(8.5, 6.0))
    shap.summary_plot(sv_pos, Xe, feature_names=feature_names, show=False)

    # ✅ NO TITLE
    fig.tight_layout()

    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"[SAVED] {out_path}")

# -------------------------
# Variants
# -------------------------
VARIANTS: List[Dict[str, str]] = [
    {"tag": "orig13",  "mode": "orig13"},   # original 13 only
    {"tag": "edgeON",  "mode": "edgeON"},   # winner features as-is
    {"tag": "edgeOFF", "mode": "edgeOFF"},  # winner features with edge removed (+policy)
]

# -------------------------
# Main (ONLY F / OF winners)
# -------------------------
for p in [RESULTS_F_PATH, RESULTS_OF_PATH, FEATURES_USED_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing file: {p}")

for dag, paths in DATA_DAG.items():
    for split, p in paths.items():
        if not os.path.exists(p):
            raise FileNotFoundError(f"Missing {dag} {split}: {p}")

dfF = pd.read_csv(RESULTS_F_PATH)
dfOF = pd.read_csv(RESULTS_OF_PATH)
feat_used = pd.read_csv(FEATURES_USED_PATH)

bestF = pick_best_row(dfF)
bestOF = pick_best_row(dfOF)

print("\n=== WINNERS (F / OF only) ===")
print("[F ]", bestF[["DAG","MODEL","K_EDGE","N_FEAT","AUPRC","ECE","Brier","FEATURE_KEY"]].to_dict())
print("[OF]", bestOF[["DAG","MODEL","K_EDGE","N_FEAT","AUPRC","ECE","Brier","FEATURE_KEY"]].to_dict())

for set_name, best in [("F", bestF), ("OF", bestOF)]:
    dag = str(best["DAG"]).strip()
    model_name = normalize_model_name(best["MODEL"])
    feature_key = str(best["FEATURE_KEY"]).strip()

    # Load data once per (set, dag)
    df_tr = read_csv_safely(DATA_DAG[dag]["train"], f"{dag}_train")
    df_va = read_csv_safely(DATA_DAG[dag]["val"],   f"{dag}_val")
    df_te = read_csv_safely(DATA_DAG[dag]["test"],  f"{dag}_test")
    target_col = detect_target_col(df_tr)

    # Winner feature list (for edgeON/edgeOFF modes)
    winner_cols_raw = load_feature_list(feat_used, feature_key)

    for v in VARIANTS:
        mode = v["mode"]
        tag = v["tag"]

        # Decide feature columns for this variant (TRAINING FEATURES)
        if mode == "orig13":
            dfcols = set(df_tr.columns)
            feat_cols = [c for c in ORIG_FEATURES_13 if c in dfcols]
            if len(feat_cols) == 0:
                raise ValueError(f"[FATAL] orig13: none of ORIG_FEATURES_13 exist in train columns for DAG={dag}")
            policy_tag = "orig13"

        elif mode == "edgeON":
            feat_cols = sanitize_feature_list(winner_cols_raw)
            policy_tag = "as_is"

        elif mode == "edgeOFF":
            feat_cols, policy_tag = filter_edge_features_with_policy(
                feat_cols=winner_cols_raw,
                include_edges=False,
                df_tr_cols=list(df_tr.columns),
                policy=EDGE_OFF_EMPTY_POLICY,
            )
            if policy_tag == "skipped":
                print(f"[SKIP] SET={set_name} VAR={tag} because edgeOFF made features empty (policy=skip).")
                continue

        else:
            raise ValueError(f"Unknown variant mode: {mode}")

        # Build XY
        X_tr, y_tr, X_va, y_va, X_te, y_te = make_xy(df_tr, df_va, df_te, target_col, feat_cols)

        # Train model and get threshold (for consistency with your pipeline)
        model, thr = train_model_and_threshold(model_name, X_tr, y_tr, X_va, y_va)

        # ✅ SHAP display names (ONLY label change)
        pretty_names = prettify_feature_names_for_shap(feat_cols)

        fname = (
            f"shap_summary__{tag}__{set_name}__{dag}__{model_name}"
            f"__n{len(feat_cols)}__{policy_tag}.png"
        )
        out_path = os.path.join(OUT_DIR, fname)

        print(f"\n[RUN] VAR={tag} SET={set_name} DAG={dag} MODEL={model_name} n_feat={len(feat_cols)} thr={thr:.3f} ({policy_tag})")
        run_shap_summary_only(
            model=model,
            X_background=X_tr,
            X_explain=X_te,
            feature_names=pretty_names,
            out_path=out_path,
            max_background=2000,
            max_explain=5000,
        )

print("\n[DONE] All variants saved to:", OUT_DIR)


[CWD] d:\University\3-2.5\PADA_Lab\Bank_Failure_Prediction_3\11_SHAP_XAI

=== WINNERS (F / OF only) ===
[F ] {'DAG': 'GES', 'MODEL': 'LightGBM', 'K_EDGE': 34, 'N_FEAT': 34, 'AUPRC': 0.4726856402210885, 'ECE': 0.016312961433284, 'Brier': 0.0167274341863156, 'FEATURE_KEY': 'b154670abb75bb96'}
[OF] {'DAG': 'GES', 'MODEL': 'LightGBM', 'K_EDGE': 34, 'N_FEAT': 47, 'AUPRC': 0.4890866630831582, 'ECE': 0.0167490496800233, 'Brier': 0.0167595664980781, 'FEATURE_KEY': '084da68000a6f7f1'}

[RUN] VAR=orig13 SET=F DAG=GES MODEL=LightGBM n_feat=13 thr=0.110 (orig13)
